In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from show import *
from utils import *
import pandas as pd
from datetime import datetime
import fnmatch


qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in ""


In [2]:
def make_map(files, dlon=1, dlat=1, stonyhurst=False, mu_thr=0.1):
    s = np.load('/home/ulyanov/data/solo/phi/distortion/fdt/distortion_cor.npz')
    xd, yd = s['xd'], s['yd']

    df = pd.read_csv('/home/ulyanov/data/solo/phi/wcs/fdt/disk_centers_cor.csv', skipinitialspace=True).dropna()
    dids = df['did'].to_numpy()
    xu_sun, yu_sun, ru_sun = df['xu_sun'].to_numpy(), df['yu_sun'].to_numpy(), df['ru_sun'].to_numpy()

    grid = np.mgrid[-90:90 + dlat / 2:dlat, -180:180:dlon]

    mean, coverage = np.zeros_like(grid[0]), np.zeros_like(grid[0])

    for file in files[:]:
        with fits.open(file) as hdul:
            header = hdul[0].header.copy()
            data = hdul[0].data.copy()

        did = int(file.split('_')[-1].split('.')[0])
        data = undistort(data, header, xd, yd)

        view = View.from_header(header)
        view.update(xc=xu_sun[dids == did][0], yc=yu_sun[dids == did][0], crota=view.crota + 0.25, rsun=ru_sun[dids == did][0], inplace=True)

        transform = view.to_synoptic(correct_mu=True, correct_dr=True, mu_thr=mu_thr, stonyhurst=stonyhurst) + ToSpherical()

        grid_, alpha = (~transform)(grid)
        data = interp2d(data, *grid_) * alpha

        n = (~np.isnan(data)) * (1 / alpha) ** 4
        coverage += np.nan_to_num(n)
        mean += np.nan_to_num((data - mean) * n / coverage)

    mean[coverage == 0] = np.nan
    coverage[coverage == 0] = np.nan
    return mean

In [3]:
df = pd.read_csv('/home/ulyanov/data/solo/phi/wcs/fdt/wcs.csv', skipinitialspace=True).dropna()

dates = np.array([datetime.fromisoformat(date) for date in df['date']])
car_rot = df['car_rot'].to_numpy()
hgln = df['hgln_obs'].to_numpy()

np.unique(car_rot)

array([2279, 2280, 2281, 2282, 2283, 2284, 2285, 2286, 2287, 2288, 2289,
       2290, 2291, 2292, 2293, 2294, 2295, 2296, 2297, 2298, 2299, 2300,
       2301, 2302, 2303, 2304, 2305, 2306])

In [4]:
t = np.where(car_rot == 2302)[0]

files = np.array(sorted(glob.glob('/home/ulyanov/data/solo/phi/2024_2025/*')))
files = files[t]

In [13]:
mean = make_map(files, dlon=0.5, dlat=0.5)

In [14]:
plt.figure(figsize=(10,8))
plt.imshow(mean, aspect='auto', origin='lower', vmin=-1000, vmax=1000, cmap='hmimag', interpolation='bicubic')
plt.tight_layout()

In [29]:
def show_data(data, figsize=(12,8), label=r'$B_{r}$, G', cmap='hmimag', vmin=-500, vmax=500, **kwargs):
    nx, ny = data.shape
    dx, dy = nx // 12, ny // 12

    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(data, aspect='auto', origin='lower', cmap=cmap, vmin=vmin, vmax=vmax, interpolation='bicubic', **kwargs)

    ax.set_xticks(np.arange(0, ny + 1, dy), np.arange(-180,181,30) % 360)
    ax.set_yticks(np.arange(0, nx + 1, dx), np.arange(-90,91,15))

    ax.set_xlabel('Longitude, degrees')
    ax.set_ylabel('Latitude, degrees')

    cax = ax.inset_axes((0.03, 0.1, 0.1, 0.015))
    fig.colorbar(im, cax=cax, orientation='horizontal', label=label)

    ax.grid(True, ls='--', color='black', alpha=0.2)
    plt.tight_layout()

In [30]:
show_data(mean)

In [36]:
plt.figure(figsize=(10,8))
plt.plot(np.mean(rebin(mean, 10), axis=1))